# PR Review AI — v2 Training (Unsloth)

Phase 1.5 / Plan A++: fine-tune CodeLlama-7B-Instruct with QLoRA via [Unsloth](https://github.com/unslothai/unsloth) for ~2x speedup and ~40–50% less VRAM vs the vanilla Transformers + PEFT + TRL stack used in `1_lora_fine_runing.ipynb`.

**Workflow:**
1. Cells 1–2: clone feature branch + install Unsloth stack.
2. Cells 3–5: 500-sample smoke test (verify training runs, checkpoint resume works, adapter loads via vanilla PEFT).
3. Cell 6: inference compatibility check — the critical pre-flight for the API server.
4. Cell 7: real 20h run (15K × 2 epochs). Skip Cell 3 (the smoke-test override) when doing this.

**Target runtime:** Colab Pro T4. See `docs/v2_evaluation.md` for the full plan.

## Cell 1 — Clone feature branch

Uses the `feat/unsloth-migration` branch. After smoke test passes and the branch merges to `main`, change `-b feat/unsloth-migration` to `-b main` (or remove `-b` entirely).

In [ ]:
import os, sys

REPO_DIR = "/content/pr-review-ai"
BRANCH = "feat/unsloth-migration"

if not os.path.isdir(REPO_DIR):
    !git clone -b {BRANCH} https://github.com/Zenlyst/PR_Review_AI.git {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

!git log --oneline -3

## Cell 2 — Install Unsloth + mount Drive + detect GPU

`setup()` runs the `unsloth[colab-new]` install line from `training/setup_colab.py` and prints the **resolved versions** of `unsloth / transformers / trl / peft`. **Copy that line down** — after the smoke test passes, you'll use it to pin the Unsloth git commit SHA in `training/setup_colab.py` (per `docs/BUG_TRACKING.md` mitigation #2).

In [ ]:
from training.setup_colab import setup
gpu_info = setup()
gpu_info

## Cell 3 — Smoke-test overrides (SKIP for the real 20h run)

In-memory overrides to `training.config`. Points checkpoints/logs/adapter at **separate `-smoke` directories** on Drive so they don't collide with the real v2 run's artifacts.

`SAVE_STEPS = 20` forces a checkpoint early enough to test the resume path without waiting.

In [ ]:
import training.config as cfg

cfg.TRAIN_SAMPLE_LIMIT = 500
cfg.NUM_EPOCHS = 1
cfg.SAVE_STEPS = 20
cfg.CHECKPOINT_DIR = "/content/drive/MyDrive/pr-review-ai/checkpoints-smoke"
cfg.ADAPTER_DIR    = "/content/drive/MyDrive/pr-review-ai/code-review-lora-adapter-smoke"
cfg.LOGGING_DIR    = "/content/drive/MyDrive/pr-review-ai/logs-smoke"

for d in (cfg.CHECKPOINT_DIR, cfg.ADAPTER_DIR, cfg.LOGGING_DIR):
    os.makedirs(d, exist_ok=True)

print(f"SMOKE CONFIG → samples={cfg.TRAIN_SAMPLE_LIMIT}, epochs={cfg.NUM_EPOCHS}, save_steps={cfg.SAVE_STEPS}")
print(f"              checkpoints → {cfg.CHECKPOINT_DIR}")
print(f"              adapter     → {cfg.ADAPTER_DIR}")

## Cell 4 — Train

Runs `training.train.run_training()`. For the smoke test: expect ~10–15 min on T4. Watch for:

- Unsloth banner at import (confirms the monkey-patches loaded **before** trl/transformers)
- `trainable params: ...` summary from `print_trainable_parameters()`
- Loss decreasing over the first ~20 steps
- A `checkpoint-20/` directory appearing under `CHECKPOINT_DIR` on Drive

For the **real 20h run:** skip Cell 3 and run this cell with the default 15K × 2-epoch config from `training/config.py`.

In [ ]:
from training.train import run_training
trainer = run_training()

## Cell 5 — Checkpoint-resume test (smoke test only)

After Cell 4 is **mid-training** and you see at least `checkpoint-20/` and `checkpoint-40/` on Drive:

1. **Runtime → Interrupt execution** to kill Cell 4.
2. Re-run Cell 4 (not this one).
3. Expect a log line like `🔄 Resuming from checkpoint: .../checkpoint-40` and training continuing from step 40, **not** step 0.

This cell just lists the smoke checkpoint dir so you can confirm what's on Drive before interrupting.

In [ ]:
!ls -la {cfg.CHECKPOINT_DIR}

## Cell 6 — Inference compatibility check (CRITICAL pre-flight)

This is the `docs/v2_evaluation.md` pre-flight check: **can a vanilla `peft.PeftModel.from_pretrained` load the Unsloth-trained adapter?** If yes, the API server (`api/model_service.py`) doesn't need Unsloth as a runtime dep and the migration is safe to ship.

**For the cleanest test:** restart the runtime first (Runtime → Restart session) so Unsloth's monkey-patches aren't lingering from training. Then re-run Cells 1 and 3, **skip Cell 2 and Cell 4**, and run this cell directly.

Uses `TEST_EXAMPLES[2]` (resource cleanup) because v1 passed that case — it's a known-good baseline for 'generation looks sane', not just 'it didn't crash'.

In [ ]:
# IMPORTANT: do NOT import unsloth in this cell. We're specifically
# testing the vanilla PEFT path that the API server uses.
from training.model import load_for_inference
from training.inference import generate_review, TEST_EXAMPLES

model, tokenizer = load_for_inference(
    adapter_path="/content/drive/MyDrive/pr-review-ai/code-review-lora-adapter-smoke"
)

case = TEST_EXAMPLES[2]  # resource cleanup — v1 passed this
print(f"=== {case['name']} ===")
print(generate_review(
    model, tokenizer,
    file_path=case['file_path'],
    before_code=case['before_code'],
    after_code=case['after_code'],
))

## Smoke-test pass criteria

Before kicking off the 20h run:

- [ ] Cell 4 trained 500 samples end-to-end, loss went down, adapter saved to the smoke dir.
- [ ] Cell 5 resume picked up from the latest checkpoint (not step 0).
- [ ] Cell 6 loaded the smoke adapter via vanilla PEFT and generated output.
- [ ] Resolved `unsloth/transformers/trl/peft` versions from Cell 2 copied somewhere.

## After smoke test passes

1. Edit `training/setup_colab.py` — replace `git+https://github.com/unslothai/unsloth.git` with a pinned commit SHA matching the resolved `unsloth.__version__`.
2. Delete the `-smoke` directories on Drive to reclaim space.
3. Commit on `feat/unsloth-migration`, merge to `main`.
4. Update Cell 1 to clone `main` instead of `feat/unsloth-migration`.
5. Skip Cell 3, run Cells 1 + 2 + 4 for the real 20h / 15K × 2-epoch run.